In [0]:
gold_df = spark.read.table("workspace.default.gold_fleet")

display(gold_df)

truck_id,driver_name,event_time,latitude,longitude,speed,engine_temp,fuel_level
UP80AB1234,Ramesh Yadav,2026-03-06T12:32:41.376Z,27.177384103325302,78.00278990586067,75.54228678242819,86.25,218.79
UP80AB1234,Ramesh Yadav,2026-03-06T12:32:46.376Z,27.177531088981876,78.0049142831851,4.78608154949781,75.22,218.75
UP80AB1234,Ramesh Yadav,2026-03-06T12:32:51.376Z,27.176993216591992,77.99871996023766,65.22773845769242,84.77,218.74
UP80AB1234,Ramesh Yadav,2026-03-06T12:32:56.376Z,27.1853909101118,78.00864893354965,20.670187090499113,90.79,218.71
UP80AB1234,Ramesh Yadav,2026-03-06T12:33:01.376Z,27.180459505785205,78.00266543693066,27.224501985989264,89.31,218.68
UP80AB1234,Ramesh Yadav,2026-03-06T12:33:06.376Z,27.177552581499828,78.00182914591103,32.40759177711274,90.56,218.64
UP80AB1234,Ramesh Yadav,2026-03-06T12:33:11.376Z,27.179477633772724,77.99593526255694,3.388695602826118,90.94,218.62
UP80AB1234,Ramesh Yadav,2026-03-06T12:33:16.376Z,27.185001399010922,77.98872163579796,52.203330348051274,79.04,218.59
UP80AB1234,Ramesh Yadav,2026-03-06T12:33:21.376Z,27.180918618064222,77.98911680059126,0.5026615404421303,90.33,218.54
UP80AB1234,Ramesh Yadav,2026-03-06T12:33:26.376Z,27.190553237673242,77.98103348017221,62.85753914215597,77.97,218.5


#KPI 1 — Overspeed Risk Trucks 🚨

Problem solved:
Fleet owners want to know which trucks are driven dangerously.

In [0]:
from pyspark.sql.functions import count

overspeed_kpi = (
    gold_df
    .filter("speed > 75")
    .groupBy("truck_id")
    .agg(count("*").alias("overspeed_events"))
    .orderBy("overspeed_events", ascending=False)
)

display(overspeed_kpi)

truck_id,overspeed_events
UP80AB1234,559
UP80QR8642,526
UP80CD5678,516
UP80MN1357,515
UP80IJ7890,512
UP80KL2468,511
UP80EF9012,494
UP80OP9753,488
UP80ST1122,484
UP80GH3456,482


#KPI 2 — Truck Idle Detection ⛽

Problem solved:
Truck burning fuel while not moving.

In [0]:
idle_kpi = (
    gold_df
    .filter("speed < 5")
    .groupBy("truck_id")
    .agg(count("*").alias("idle_events"))
    .orderBy("idle_events", ascending=False)
)

display(idle_kpi)

truck_id,idle_events
UP80GH3456,1617
UP80IJ7890,1603
UP80EF9012,1591
UP80AB1234,1589
UP80ST1122,1583
UP80OP9753,1554
UP80CD5678,1553
UP80QR8642,1548
UP80KL2468,1542
UP80MN1357,1540


#KPI 3 — Engine Overheating Alert 🔥

Problem solved:
Engine overheating damages trucks.

In [0]:
overheat_kpi = (
    gold_df
    .filter("engine_temp > 90")
    .groupBy("truck_id")
    .agg(count("*").alias("overheat_events"))
    .orderBy("overheat_events", ascending=False)
)

display(overheat_kpi)

truck_id,overheat_events
UP80EF9012,1053
UP80ST1122,1041
UP80AB1234,1036
UP80MN1357,1025
UP80OP9753,1015
UP80GH3456,1010
UP80KL2468,994
UP80CD5678,992
UP80QR8642,986
UP80IJ7890,970


In [0]:
from pyspark.sql.functions import avg

engine_health_kpi = (
    gold_df
    .groupBy("truck_id")
    .agg(
        avg("engine_temp").alias("avg_engine_temp")
    )
    .orderBy("avg_engine_temp", ascending=False)
)

display(engine_health_kpi)

truck_id,avg_engine_temp
UP80MN1357,82.60363999999983
UP80ST1122,82.56039400000002
UP80OP9753,82.55746600000003
UP80AB1234,82.54959999999991
UP80GH3456,82.53559400000023
UP80CD5678,82.517242
UP80EF9012,82.49104199999998
UP80IJ7890,82.48834400000028
UP80KL2468,82.45144400000008
UP80QR8642,82.36280800000013


#KPI 4 — Fuel Consumption per Truck ⛽

In [0]:
from pyspark.sql.functions import min, max, col

fuel_kpi = (
    gold_df
    .groupBy("truck_id")
    .agg(
        max("fuel_level").alias("max_fuel"),
        min("fuel_level").alias("min_fuel")
    )
    .withColumn(
        "fuel_consumed",
        col("max_fuel") - col("min_fuel")
    )
)

display(fuel_kpi)

truck_id,max_fuel,min_fuel,fuel_consumed
UP80AB1234,218.79,68.76,150.02999999999997
UP80CD5678,244.8,95.8,149.0
UP80EF9012,291.81,141.23,150.58
UP80GH3456,206.37,55.38,150.99
UP80IJ7890,246.32,97.41,148.91
UP80KL2468,237.06,86.41,150.65
UP80MN1357,289.6,139.43,150.17000000000002
UP80OP9753,266.89,115.88,151.01
UP80QR8642,209.42,59.6,149.82
UP80ST1122,208.1,57.69,150.41


#KPI 5 — Average Speed per Driver 🚚

Fleet managers use this to evaluate driver performance.

In [0]:
from pyspark.sql.functions import avg

driver_speed_kpi = (
    gold_df
    .groupBy("driver_name")
    .agg(
        avg("speed").alias("avg_driver_speed")
    )
)

display(driver_speed_kpi)

driver_name,avg_driver_speed
Ramesh Yadav,35.4981100051955
Amit Sharma,35.88329948581583
Sandeep Singh,35.35809854883906
Vikram Chauhan,34.870027304232735
Rahul Verma,35.109223715628296
Deepak Kumar,35.7904387916296
Pankaj Gupta,35.98499010907986
Anil Mishra,35.686856818644614
Manoj Tiwari,35.741320323183864
Arjun Pandey,35.18562553118341


#KPI 6 — Distance Travelled per Truck 📍

(Approximation using speed + time)

In [0]:
from pyspark.sql.functions import lag
from pyspark.sql.window import Window

window = Window.partitionBy("truck_id").orderBy("event_time")

distance_df = gold_df.withColumn(
    "prev_time",
    lag("event_time").over(window)
)

distance_df = distance_df.withColumn(
    "time_diff_hours",
    (col("event_time").cast("long") - col("prev_time").cast("long")) / 3600
)

distance_df = distance_df.withColumn(
    "distance_km",
    col("speed") * col("time_diff_hours")
)

distance_kpi = distance_df.groupBy("truck_id").sum("distance_km")

display(distance_kpi)

truck_id,sum(distance_km)
UP80AB1234,246.4097329711041
UP80CD5678,249.1509880219864
UP80EF9012,245.45736587413566
UP80GH3456,242.15184507176264
UP80IJ7890,243.7234786423338
UP80KL2468,248.50539272575563
UP80MN1357,249.84682255003713
UP80OP9753,247.82375416529217
UP80QR8642,248.10970121256094
UP80ST1122,244.33795778436075
